In [ ]:
````xml
<!-- filepath: c:\Users\ADMIN\Downloads\mimic-preprocessing-main\mimic-preprocessing-main\explainable_medical_diagnosis_demo.ipynb -->
<VSCode.Cell language="markdown">
# Explainable Medical Diagnosis with SHAP and LIME on MIMIC-III Data

This comprehensive notebook demonstrates how to build explainable AI systems for medical diagnosis using the MIMIC-III dataset from Kaggle. We'll cover:

1. **Data Loading & Preprocessing** - Load MIMIC-III data and prepare features
2. **Model Training** - Train ML models for medical diagnosis prediction
3. **SHAP Analysis** - Global and local interpretability with SHAP
4. **LIME Explanations** - Individual patient explanations
5. **Clinical Validation** - Assess clinical relevance of explanations
6. **Interactive Dashboard** - Present results for clinical use

## Why Explainability Matters in Healthcare

In medical diagnosis, it's not enough to have accurate predictions. Clinicians need to understand:
- **Why** a patient is classified as high-risk
- **Which features** contribute most to the prediction
- **How reliable** the model's reasoning is
- **Whether** the model aligns with clinical knowledge

SHAP and LIME provide complementary approaches to model interpretability:
- **SHAP**: Game-theory based, consistent feature attributions, global insights
- **LIME**: Local approximations, intuitive explanations, individual focus
</VSCode.Cell>

<VSCode.Cell language="python">
# Setup and Import Libraries
import warnings
warnings.filterwarnings('ignore')

# Data handling
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_curve, roc_curve,
    classification_report, confusion_matrix, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb

# Explainability libraries
import shap
import lime
from lime import lime_tabular

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Kaggle dataset download
try:
    import kagglehub
    print("✅ kagglehub available - can download from Kaggle")
except ImportError:
    print("❌ kagglehub not installed. Run: pip install kagglehub")

print("✅ All libraries loaded successfully!")
print(f"📊 SHAP version: {shap.__version__}")
print(f"🔍 LIME version: {lime.__version__}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Data Loading and Preprocessing

We'll start by downloading the MIMIC-III dataset from Kaggle and loading the key tables needed for our analysis.
</VSCode.Cell>

<VSCode.Cell language="python">
# Download MIMIC-III dataset from Kaggle
print("🔄 Downloading MIMIC-III dataset from Kaggle...")
try:
    # Download the dataset
    mimic_path = kagglehub.dataset_download("asjad99/mimiciii")
    print(f"✅ Dataset downloaded to: {mimic_path}")
    
    # Convert to Path object for easier handling
    mimic_dir = Path(mimic_path)
    
    # List available CSV files
    csv_files = list(mimic_dir.glob("*.csv"))
    print(f"\n📁 Found {len(csv_files)} CSV files:")
    for file in sorted(csv_files):
        file_size = file.stat().st_size / (1024*1024)  # Size in MB
        print(f"  📄 {file.name} ({file_size:.1f} MB)")
        
except Exception as e:
    print(f"❌ Error downloading dataset: {e}")
    print("🔧 Please ensure you have kagglehub installed and proper authentication")
    # Fallback - use sample data structure for demonstration
    mimic_dir = Path("./sample_mimic_data")
    print(f"📝 Using fallback sample data directory: {mimic_dir}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Load key MIMIC-III tables
print("📊 Loading key MIMIC-III tables...")

try:
    # Core demographic and administrative tables
    print("  Loading PATIENTS...")
    patients = pd.read_csv(mimic_dir / "PATIENTS.csv")
    
    print("  Loading ADMISSIONS...")
    admissions = pd.read_csv(mimic_dir / "ADMISSIONS.csv")
    
    print("  Loading ICUSTAYS...")
    icustays = pd.read_csv(mimic_dir / "ICUSTAYS.csv")
    
    # Clinical data tables (sample first few rows for memory efficiency)
    print("  Loading CHARTEVENTS (first 100k rows)...")
    chartevents = pd.read_csv(mimic_dir / "CHARTEVENTS.csv", nrows=100000)
    
    print("  Loading LABEVENTS (first 100k rows)...")
    labevents = pd.read_csv(mimic_dir / "LABEVENTS.csv", nrows=100000)
    
    print("  Loading DIAGNOSES_ICD...")
    diagnoses = pd.read_csv(mimic_dir / "DIAGNOSES_ICD.csv")
    
    # Display basic information
    print("\n📈 Dataset Summary:")
    print(f"  👥 Patients: {patients.shape[0]:,} records")
    print(f"  🏥 Admissions: {admissions.shape[0]:,} records")
    print(f"  🛏️ ICU Stays: {icustays.shape[0]:,} records")
    print(f"  📊 Chart Events: {chartevents.shape[0]:,} records (sample)")
    print(f"  🧪 Lab Events: {labevents.shape[0]:,} records (sample)")
    print(f"  🏷️ Diagnoses: {diagnoses.shape[0]:,} records")
    
    data_loaded = True
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("🔧 Creating synthetic demo data for illustration...")
    data_loaded = False
</VSCode.Cell>

<VSCode.Cell language="python">
# Create synthetic data for demonstration if real data not available
if not data_loaded:
    print("🔄 Creating synthetic MIMIC-III-like dataset for demonstration...")
    
    np.random.seed(42)
    n_patients = 1000
    
    # Create synthetic patient data
    patients = pd.DataFrame({
        'SUBJECT_ID': range(1, n_patients + 1),
        'GENDER': np.random.choice(['M', 'F'], n_patients),
        'DOB': pd.date_range('1920-01-01', '2000-01-01', periods=n_patients),
        'DOD': [None] * n_patients  # Most patients alive
    })
    
    # Create synthetic admissions
    admissions = pd.DataFrame({
        'SUBJECT_ID': np.random.choice(range(1, n_patients + 1), n_patients),
        'HADM_ID': range(10000, 10000 + n_patients),
        'ADMISSION_TYPE': np.random.choice(['EMERGENCY', 'ELECTIVE', 'URGENT'], n_patients),
        'HOSPITAL_EXPIRE_FLAG': np.random.choice([0, 1], n_patients, p=[0.85, 0.15]),
        'ADMITTIME': pd.date_range('2010-01-01', '2019-12-31', periods=n_patients),
        'ETHNICITY': np.random.choice(['WHITE', 'BLACK', 'HISPANIC', 'ASIAN', 'OTHER'], n_patients),
        'INSURANCE': np.random.choice(['Medicare', 'Medicaid', 'Private'], n_patients)
    })
    
    # Create synthetic vital signs and lab values
    feature_data = pd.DataFrame({
        'SUBJECT_ID': admissions['SUBJECT_ID'],
        'HADM_ID': admissions['HADM_ID'],
        'AGE': np.random.normal(65, 15, n_patients).clip(18, 100),
        'HEART_RATE_MEAN': np.random.normal(80, 15, n_patients).clip(40, 150),
        'BLOOD_PRESSURE_SYSTOLIC_MEAN': np.random.normal(120, 20, n_patients).clip(80, 200),
        'TEMPERATURE_MEAN': np.random.normal(98.6, 1.5, n_patients).clip(95, 105),
        'GLUCOSE_MEAN': np.random.normal(120, 40, n_patients).clip(70, 400),
        'CREATININE_MEAN': np.random.normal(1.2, 0.8, n_patients).clip(0.5, 5),
        'WHITE_BLOOD_CELL_COUNT': np.random.normal(8, 3, n_patients).clip(2, 20),
        'HEMOGLOBIN': np.random.normal(12, 2, n_patients).clip(6, 18),
        'PLATELET_COUNT': np.random.normal(250, 100, n_patients).clip(50, 500)
    })
    
    print("✅ Synthetic dataset created successfully!")
    print(f"  👥 Synthetic Patients: {len(patients):,}")
    print(f"  🏥 Synthetic Admissions: {len(admissions):,}")
    print(f"  📊 Synthetic Features: {len(feature_data):,}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Feature Engineering for Medical Diagnosis

Now we'll create a comprehensive feature set that combines demographic, clinical, and temporal information for diagnosis prediction.
</VSCode.Cell>

<VSCode.Cell language="python">
# Feature Engineering for Medical Diagnosis Prediction
print("🔧 Engineering features for medical diagnosis prediction...")

def create_diagnosis_features(admissions_df, patients_df, feature_data_df):
    """
    Create a comprehensive feature set for medical diagnosis prediction.
    """
    print("  🔄 Merging patient demographics...")
    
    # Merge admissions with patients
    df = admissions_df.merge(patients_df[['SUBJECT_ID', 'GENDER', 'DOB']], 
                            on='SUBJECT_ID', how='left')
    
    # Calculate age at admission
    if data_loaded:
        df['ADMITTIME'] = pd.to_datetime(df['ADMITTIME'])
        df['DOB'] = pd.to_datetime(df['DOB'])
        df['AGE'] = (df['ADMITTIME'] - df['DOB']).dt.days / 365.25
    else:
        # For synthetic data, age is already calculated
        df = df.merge(feature_data_df[['SUBJECT_ID', 'AGE']], on='SUBJECT_ID', how='left')
    
    print("  🔄 Adding clinical features...")
    
    # Add clinical measurements if available
    if 'feature_data_df' in locals() or not data_loaded:
        clinical_cols = [col for col in feature_data_df.columns 
                        if col not in ['SUBJECT_ID', 'HADM_ID', 'AGE']]
        df = df.merge(feature_data_df[['SUBJECT_ID'] + clinical_cols], 
                     on='SUBJECT_ID', how='left')
    
    print("  🔄 Encoding categorical variables...")
    
    # Encode categorical variables
    categorical_features = ['GENDER', 'ADMISSION_TYPE', 'ETHNICITY', 'INSURANCE']
    label_encoders = {}
    
    for feature in categorical_features:
        if feature in df.columns:
            le = LabelEncoder()
            df[f'{feature}_ENCODED'] = le.fit_transform(df[feature].fillna('UNKNOWN'))
            label_encoders[feature] = le
    
    print("  🔄 Creating risk indicators...")
    
    # Create risk indicators
    df['AGE_HIGH_RISK'] = (df['AGE'] > 70).astype(int)
    df['EMERGENCY_ADMISSION'] = (df['ADMISSION_TYPE'] == 'EMERGENCY').astype(int)
    
    if not data_loaded:
        # Add some clinical risk indicators for synthetic data
        df['HIGH_GLUCOSE'] = (df['GLUCOSE_MEAN'] > 150).astype(int)
        df['HIGH_CREATININE'] = (df['CREATININE_MEAN'] > 1.5).astype(int)
        df['LOW_HEMOGLOBIN'] = (df['HEMOGLOBIN'] < 10).astype(int)
    
    return df, label_encoders

# Create the feature dataset
df_features, encoders = create_diagnosis_features(admissions, patients, feature_data)

print(f"✅ Feature engineering completed!")
print(f"  📊 Final dataset shape: {df_features.shape}")
print(f"  🔢 Available features: {df_features.columns.tolist()}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Prepare dataset for modeling
print("🎯 Preparing dataset for mortality prediction modeling...")

# Define target variable (in-hospital mortality)
target_col = 'HOSPITAL_EXPIRE_FLAG'
y = df_features[target_col].copy()

# Define feature columns for modeling
feature_columns = [
    'AGE', 'GENDER_ENCODED', 'ADMISSION_TYPE_ENCODED', 
    'ETHNICITY_ENCODED', 'INSURANCE_ENCODED',
    'AGE_HIGH_RISK', 'EMERGENCY_ADMISSION'
]

# Add clinical features if available
if not data_loaded:
    clinical_features = [
        'HEART_RATE_MEAN', 'BLOOD_PRESSURE_SYSTOLIC_MEAN', 'TEMPERATURE_MEAN',
        'GLUCOSE_MEAN', 'CREATININE_MEAN', 'WHITE_BLOOD_CELL_COUNT',
        'HEMOGLOBIN', 'PLATELET_COUNT', 'HIGH_GLUCOSE', 'HIGH_CREATININE', 'LOW_HEMOGLOBIN'
    ]
    feature_columns.extend(clinical_features)

# Select features that exist in the dataset
available_features = [col for col in feature_columns if col in df_features.columns]
X = df_features[available_features].copy()

# Handle missing values
X = X.fillna(X.median())
y = y.fillna(0)  # Assume no mortality if missing

# Remove any rows with missing target
mask = ~y.isna()
X = X[mask]
y = y[mask]

print(f"✅ Dataset prepared for modeling:")
print(f"  📊 Features shape: {X.shape}")
print(f"  🎯 Target distribution: {y.value_counts().to_dict()}")
print(f"  📋 Feature list: {X.columns.tolist()}")

# Display basic statistics
print(f"\n📈 Target Variable Statistics:")
print(f"  💀 Mortality rate: {y.mean():.1%}")
print(f"  ✅ Survival rate: {(1-y.mean()):.1%}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Train Machine Learning Models for Diagnosis Prediction

We'll train multiple models and select the best performing one for our explainability analysis.
</VSCode.Cell>

<VSCode.Cell language="python">
# Split data into training and testing sets
print("🔄 Splitting data into train/test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print(f"✅ Data split completed:")
print(f"  🎓 Training set: {X_train.shape[0]} samples")
print(f"  🧪 Test set: {X_test.shape[0]} samples")
print(f"  📊 Training mortality rate: {y_train.mean():.1%}")
print(f"  📊 Test mortality rate: {y_test.mean():.1%}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Train multiple models and compare performance
print("🤖 Training multiple machine learning models...")

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, 
                                random_state=42, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, 
                                  random_state=42, verbose=-1),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

model_results = {}

for name, model in models.items():
    print(f"  🔄 Training {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    
    model_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'auc': auc,
        'f1_score': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"    ✅ {name} - AUC: {auc:.3f}, Accuracy: {accuracy:.3f}, F1: {f1:.3f}")

# Select best model based on AUC
best_model_name = max(model_results.keys(), key=lambda k: model_results[k]['auc'])
best_model = model_results[best_model_name]['model']

print(f"\n🏆 Best model: {best_model_name}")
print(f"  📊 AUC: {model_results[best_model_name]['auc']:.3f}")
print(f"  📊 Accuracy: {model_results[best_model_name]['accuracy']:.3f}")
print(f"  📊 F1-Score: {model_results[best_model_name]['f1_score']:.3f}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Create model performance comparison visualization
print("📊 Creating model performance visualization...")

# Prepare data for plotting
model_names = list(model_results.keys())
metrics = ['accuracy', 'auc', 'f1_score']
metric_labels = ['Accuracy', 'AUC', 'F1-Score']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['skyblue', 'lightgreen', 'lightcoral', 'gold']

for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    values = [model_results[name][metric] for name in model_names]
    
    bars = axes[i].bar(model_names, values, color=colors)
    axes[i].set_title(f'{label} Comparison')
    axes[i].set_ylabel(label)
    axes[i].set_ylim(0, 1)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{value:.3f}', ha='center', va='bottom')
    
    # Rotate x-axis labels if needed
    plt.setp(axes[i].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f"✅ Model comparison completed. Best model: {best_model_name}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. SHAP Analysis - Global and Local Interpretability

Now we'll use SHAP to understand how our best model makes predictions, both globally (across all patients) and locally (for individual patients).
</VSCode.Cell>

<VSCode.Cell language="python">
# Setup SHAP explainer for the best model
print(f"🔍 Setting up SHAP explainer for {best_model_name}...")

# Initialize SHAP explainer based on model type
if best_model_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    explainer = shap.TreeExplainer(best_model)
    print("  ✅ Using TreeExplainer for tree-based model")
else:
    # Use KernelExplainer for other models (slower but more general)
    explainer = shap.KernelExplainer(
        best_model.predict_proba,
        X_train_scaled.sample(100, random_state=42)  # Background dataset
    )
    print("  ✅ Using KernelExplainer for non-tree model")

# Compute SHAP values for test set (limit to first 200 for speed)
print("  🔄 Computing SHAP values...")
n_explain = min(200, len(X_test_scaled))
X_explain = X_test_scaled.iloc[:n_explain]

shap_values = explainer.shap_values(X_explain)

# For binary classification, get positive class SHAP values
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Positive class (mortality)

print(f"✅ SHAP values computed for {n_explain} test samples")
print(f"  📊 SHAP values shape: {shap_values.shape}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Create SHAP summary plot - Global feature importance
print("📊 Creating SHAP summary plots...")

# Summary plot showing feature importance and impact
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_explain, feature_names=list(X.columns), show=False)
plt.title('SHAP Summary Plot - Feature Impact on Mortality Prediction')
plt.tight_layout()
plt.show()

print("✅ SHAP summary plot created")
</VSCode.Cell>

<VSCode.Cell language="python">
# Create SHAP feature importance bar plot
print("📊 Creating SHAP feature importance plot...")

# Calculate mean absolute SHAP values for feature importance
feature_importance = np.abs(shap_values).mean(0)
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=True)

# Create horizontal bar plot
plt.figure(figsize=(10, 8))
bars = plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Mean |SHAP Value|')
plt.title('Global Feature Importance (SHAP)')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for bar, value in zip(bars, importance_df['importance']):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f'{value:.3f}', va='center', ha='left')

plt.tight_layout()
plt.show()

print("✅ SHAP feature importance plot created")
print("\n🔍 Top 5 most important features:")
for i, (_, row) in enumerate(importance_df.tail(5).iterrows(), 1):
    print(f"  {i}. {row['feature']}: {row['importance']:.3f}")
</VSCode.Cell>

<VSCode.Cell language="python">
# SHAP waterfall plot for individual patient explanation
print("🔍 Creating SHAP waterfall plot for individual patient...")

# Select a high-risk patient for detailed explanation
patient_idx = 0  # First patient in test set
patient_data = X_explain.iloc[patient_idx]
patient_shap = shap_values[patient_idx]
patient_prediction = best_model.predict_proba([patient_data.values])[0, 1]

print(f"  👤 Patient {patient_idx}: Risk Score = {patient_prediction:.3f}")
print(f"  🎯 Actual Outcome: {'Mortality' if y_test.iloc[patient_idx] else 'Survival'}")

# Create waterfall plot
plt.figure(figsize=(12, 8))

# Sort features by absolute SHAP value
shap_df = pd.DataFrame({
    'feature': X.columns,
    'shap_value': patient_shap,
    'feature_value': patient_data.values
}).sort_key = lambda x: abs(x)
shap_df = shap_df.reindex(shap_df['shap_value'].abs().sort_values(ascending=False).index)

# Take top 10 features
top_features = shap_df.head(10)

# Create horizontal bar plot
colors = ['red' if x < 0 else 'green' for x in top_features['shap_value']]
bars = plt.barh(range(len(top_features)), top_features['shap_value'], color=colors, alpha=0.7)

# Customize plot
plt.yticks(range(len(top_features)), 
          [f"{row['feature']}\n= {row['feature_value']:.2f}" 
           for _, row in top_features.iterrows()])
plt.xlabel('SHAP Value (Impact on Prediction)')
plt.title(f'SHAP Explanation for Patient {patient_idx}\n'
          f'Risk Score: {patient_prediction:.3f} | '
          f'Actual: {"Mortality" if y_test.iloc[patient_idx] else "Survival"}')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, value) in enumerate(zip(bars, top_features['shap_value'])):
    plt.text(bar.get_width() + (0.001 if value >= 0 else -0.001), 
             bar.get_y() + bar.get_height()/2,
             f'{value:.3f}', va='center', 
             ha='left' if value >= 0 else 'right')

plt.tight_layout()
plt.show()

print("✅ SHAP waterfall plot created for individual patient")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. LIME Analysis - Local Interpretable Model-Agnostic Explanations

LIME provides a different perspective on interpretability by creating local linear approximations around individual predictions.
</VSCode.Cell>

<VSCode.Cell language="python">
# Setup LIME explainer
print("🔍 Setting up LIME explainer...")

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=list(X.columns),
    class_names=['Survival', 'Mortality'],
    mode='classification',
    discretize_continuous=True
)

print("✅ LIME explainer initialized")
</VSCode.Cell>

<VSCode.Cell language="python">
# Create LIME explanation for the same patient we analyzed with SHAP
print(f"🔍 Creating LIME explanation for Patient {patient_idx}...")

# Generate LIME explanation
patient_data_lime = X_test_scaled.iloc[patient_idx].values
lime_exp = lime_explainer.explain_instance(
    patient_data_lime,
    best_model.predict_proba,
    num_features=10,
    top_labels=2
)

# Extract explanation data
lime_list = lime_exp.as_list()
lime_features, lime_contributions = zip(*lime_list)

print(f"✅ LIME explanation generated")
print(f"  📊 Top {len(lime_list)} feature contributions:")
for feature, contribution in lime_list:
    print(f"    {feature}: {contribution:+.3f}")
</VSCode.Cell>

<VSCode.Cell language="python">
# Visualize LIME explanation
print("📊 Creating LIME explanation visualization...")

plt.figure(figsize=(12, 8))

# Create horizontal bar plot for LIME contributions
colors = ['red' if x < 0 else 'green' for x in lime_contributions]
bars = plt.barh(range(len(lime_features)), lime_contributions, color=colors, alpha=0.7)

# Customize plot
plt.yticks(range(len(lime_features)), lime_features)
plt.xlabel('Feature Contribution to Prediction')
plt.title(f'LIME Explanation for Patient {patient_idx}\n'
          f'Risk Score: {patient_prediction:.3f} | '
          f'Actual: {"Mortality" if y_test.iloc[patient_idx] else "Survival"}')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for bar, value in zip(bars, lime_contributions):
    plt.text(bar.get_width() + (0.01 if value >= 0 else -0.01), 
             bar.get_y() + bar.get_height()/2,
             f'{value:+.3f}', va='center', 
             ha='left' if value >= 0 else 'right')

plt.tight_layout()
plt.show()

print("✅ LIME explanation visualization created")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 6. Compare SHAP vs LIME Explanations

Let's compare how SHAP and LIME explain the same patient's prediction to understand the similarities and differences between these approaches.
</VSCode.Cell>

<VSCode.Cell language="python">
# Compare SHAP and LIME explanations side by side
print("🔄 Comparing SHAP vs LIME explanations...")

# Prepare SHAP data for comparison
shap_df = pd.DataFrame({
    'feature': X.columns,
    'shap_value': patient_shap
}).sort_values('shap_value', key=abs, ascending=False).head(10)

# Prepare LIME data for comparison
lime_df = pd.DataFrame({
    'feature': lime_features,
    'lime_value': lime_contributions
})

# Create side-by-side comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# SHAP plot
shap_colors = ['red' if x < 0 else 'green' for x in shap_df['shap_value']]
bars1 = ax1.barh(range(len(shap_df)), shap_df['shap_value'], color=shap_colors, alpha=0.7)
ax1.set_yticks(range(len(shap_df)))
ax1.set_yticklabels(shap_df['feature'])
ax1.set_xlabel('SHAP Value')
ax1.set_title('SHAP Explanation')
ax1.grid(axis='x', alpha=0.3)

# Add SHAP value labels
for bar, value in zip(bars1, shap_df['shap_value']):
    ax1.text(bar.get_width() + (0.001 if value >= 0 else -0.001), 
             bar.get_y() + bar.get_height()/2,
             f'{value:+.3f}', va='center', 
             ha='left' if value >= 0 else 'right')

# LIME plot
lime_colors = ['red' if x < 0 else 'green' for x in lime_df['lime_value']]
bars2 = ax2.barh(range(len(lime_df)), lime_df['lime_value'], color=lime_colors, alpha=0.7)
ax2.set_yticks(range(len(lime_df)))
ax2.set_yticklabels([f.split(' ')[0] for f in lime_df['feature']])  # Simplify feature names
ax2.set_xlabel('LIME Contribution')
ax2.set_title('LIME Explanation')
ax2.grid(axis='x', alpha=0.3)

# Add LIME value labels
for bar, value in zip(bars2, lime_df['lime_value']):
    ax2.text(bar.get_width() + (0.01 if value >= 0 else -0.01), 
             bar.get_y() + bar.get_height()/2,
             f'{value:+.3f}', va='center', 
             ha='left' if value >= 0 else 'right')

plt.suptitle(f'SHAP vs LIME Comparison for Patient {patient_idx}\n'
             f'Risk Score: {patient_prediction:.3f}', fontsize=16)
plt.tight_layout()
plt.show()

print("✅ SHAP vs LIME comparison visualization created")
</VSCode.Cell>

<VSCode.Cell language="python">
# Analyze explanation consistency between SHAP and LIME
print("🔍 Analyzing explanation consistency between SHAP and LIME...")

# Find common features between SHAP and LIME explanations
shap_features = set(shap_df['feature'].head(5))
lime_feature_names = set([f.split(' ')[0] for f in lime_df['feature'].head(5)])

common_features = shap_features.intersection(lime_feature_names)
print(f"  🤝 Common top features: {common_features}")

# Calculate correlation between SHAP and LIME for common features
if len(common_features) > 1:
    shap_common = shap_df[shap_df['feature'].isin(common_features)]['shap_value']
    # Note: LIME feature names include conditions, so we need to match carefully
    lime_common_values = []
    for feature in common_features:
        matching_lime = lime_df[lime_df['feature'].str.contains(feature, case=False)]
        if not matching_lime.empty:
            lime_common_values.append(matching_lime['lime_value'].iloc[0])
    
    if len(lime_common_values) == len(shap_common):
        correlation = np.corrcoef(shap_common, lime_common_values)[0, 1]
        print(f"  📊 Correlation between SHAP and LIME (common features): {correlation:.3f}")
    else:
        print("  ⚠️ Could not compute correlation - feature matching issues")
else:
    print("  ⚠️ Not enough common features for correlation analysis")

# Explanation consistency analysis
print(f"\n📝 Explanation Analysis:")
print(f"  🎯 Patient prediction: {patient_prediction:.3f} ({'High Risk' if patient_prediction > 0.5 else 'Low Risk'})")
print(f"  🔍 SHAP explanation focuses on: {shap_df.iloc[0]['feature']} (impact: {shap_df.iloc[0]['shap_value']:+.3f})")
print(f"  🔍 LIME explanation focuses on: {lime_features[0]} (impact: {lime_contributions[0]:+.3f})")

# Agreement in direction (positive vs negative impact)
shap_direction = "increases" if shap_df.iloc[0]['shap_value'] > 0 else "decreases"
lime_direction = "increases" if lime_contributions[0] > 0 else "decreases"
agreement = shap_direction == lime_direction

print(f"  ✅ Both methods agree the top feature {shap_direction} risk: {agreement}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 7. Clinical Validation and Interpretation

Now let's assess whether our model explanations align with clinical knowledge and identify any potential concerns.
</VSCode.Cell>

<VSCode.Cell language="python">
# Clinical validation of model explanations
print("🏥 Performing clinical validation of model explanations...")

# Define clinically expected relationships for mortality prediction
clinical_expectations = {
    'AGE': 'positive',  # Higher age should increase mortality risk
    'EMERGENCY_ADMISSION': 'positive',  # Emergency admissions are higher risk
    'HIGH_CREATININE': 'positive',  # Kidney dysfunction increases risk
    'LOW_HEMOGLOBIN': 'positive',  # Anemia can indicate serious conditions
    'HIGH_GLUCOSE': 'positive',  # Diabetes complications increase risk
    'HEART_RATE_MEAN': 'context',  # Very high or very low can be concerning
    'BLOOD_PRESSURE_SYSTOLIC_MEAN': 'context',  # Both high and low can be risky
}

print("✅ Clinical expectations defined")

# Analyze global feature importance against clinical expectations
print("\n🔍 Analyzing global feature importance against clinical knowledge:")

clinical_alignment = {}
for feature, expected_direction in clinical_expectations.items():
    if feature in importance_df['feature'].values:
        importance = importance_df[importance_df['feature'] == feature]['importance'].iloc[0]
        
        # For features in our model, check average SHAP direction
        feature_idx = list(X.columns).index(feature)
        avg_shap = np.mean(shap_values[:, feature_idx])
        actual_direction = 'positive' if avg_shap > 0 else 'negative'
        
        if expected_direction == 'context':
            alignment = 'contextual'  # These features are complex
        else:
            alignment = 'aligned' if actual_direction == expected_direction else 'misaligned'
        
        clinical_alignment[feature] = {
            'importance': importance,
            'expected': expected_direction,
            'actual': actual_direction,
            'avg_shap': avg_shap,
            'alignment': alignment
        }
        
        status_icon = "✅" if alignment == 'aligned' else "⚠️" if alignment == 'contextual' else "❌"
        print(f"  {status_icon} {feature}: Expected {expected_direction}, "
              f"Actual {actual_direction} (avg SHAP: {avg_shap:+.3f})")

print(f"\n📊 Clinical Alignment Summary:")
aligned_count = sum(1 for v in clinical_alignment.values() if v['alignment'] == 'aligned')
total_count = len(clinical_alignment)
print(f"  ✅ Aligned features: {aligned_count}/{total_count} ({aligned_count/total_count:.1%})")
</VSCode.Cell>

<VSCode.Cell language="python">
# Identify potential model concerns
print("🚨 Identifying potential model concerns...")

concerns = []

# Check for unexpected feature importance
print("\n🔍 Checking for unexpected feature patterns:")

for feature, info in clinical_alignment.items():
    if info['alignment'] == 'misaligned':
        concerns.append(f"Feature '{feature}' shows unexpected relationship with mortality")
        print(f"  ⚠️ {feature}: Expected {info['expected']} but shows {info['actual']} relationship")

# Check for demographic bias
demographic_features = ['GENDER_ENCODED', 'ETHNICITY_ENCODED', 'INSURANCE_ENCODED']
print(f"\n🔍 Checking for demographic bias:")

for feature in demographic_features:
    if feature in X.columns:
        feature_idx = list(X.columns).index(feature)
        importance = importance_df[importance_df['feature'] == feature]['importance'].iloc[0]
        
        if importance > 0.05:  # Threshold for concerning importance
            concerns.append(f"Demographic feature '{feature}' has high importance ({importance:.3f})")
            print(f"  ⚠️ {feature}: High importance ({importance:.3f}) - potential bias concern")
        else:
            print(f"  ✅ {feature}: Low importance ({importance:.3f}) - no bias concern")

# Check for data leakage indicators
print(f"\n🔍 Checking for potential data leakage:")
high_importance_features = importance_df[importance_df['importance'] > 0.1]['feature'].tolist()

for feature in high_importance_features:
    # Check if feature name suggests post-outcome measurement
    if any(keyword in feature.lower() for keyword in ['discharge', 'expire', 'death', 'outcome']):
        concerns.append(f"Feature '{feature}' may indicate data leakage")
        print(f"  ⚠️ {feature}: May indicate data leakage")

if not concerns:
    print("  ✅ No obvious data leakage concerns detected")

# Summary of concerns
print(f"\n📋 Model Validation Summary:")
if concerns:
    print(f"  ⚠️ {len(concerns)} potential concerns identified:")
    for i, concern in enumerate(concerns, 1):
        print(f"    {i}. {concern}")
else:
    print("  ✅ No major concerns identified in clinical validation")

print(f"\n💡 Recommendations:")
print(f"  1. Validate feature relationships with clinical experts")
print(f"  2. Monitor model performance over time")
print(f"  3. Consider additional feature engineering for complex relationships")
print(f"  4. Implement fairness constraints if demographic bias is detected")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 8. Create Clinical Explanation Dashboard

Finally, let's create a comprehensive clinical report that presents our findings in a format suitable for healthcare professionals.
</VSCode.Cell>

<VSCode.Cell language="python">
# Create comprehensive clinical report
print("📋 Creating comprehensive clinical explanation report...")

def generate_clinical_report(patient_idx, model, X_test, y_test, shap_values, lime_exp):
    """Generate a comprehensive clinical report for a patient."""
    
    # Basic prediction information
    patient_data = X_test.iloc[patient_idx]
    prediction_proba = model.predict_proba([patient_data.values])[0, 1]
    actual_outcome = y_test.iloc[patient_idx]
    
    # Risk categorization
    if prediction_proba >= 0.7:
        risk_category = "HIGH RISK"
        risk_color = "🔴"
    elif prediction_proba >= 0.3:
        risk_category = "MODERATE RISK"
        risk_color = "🟡"
    else:
        risk_category = "LOW RISK"
        risk_color = "🟢"
    
    # SHAP contributions
    patient_shap = shap_values[patient_idx]
    shap_contributions = list(zip(X.columns, patient_shap))
    shap_contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    
    # Clinical recommendations based on risk level
    if prediction_proba >= 0.7:
        recommendations = [
            "Immediate clinical assessment recommended",
            "Consider ICU monitoring or increased observation frequency",
            "Review vital signs and lab values closely",
            "Evaluate need for additional interventions"
        ]
    elif prediction_proba >= 0.3:
        recommendations = [
            "Enhanced monitoring recommended",
            "Regular assessment of clinical status",
            "Consider preventive interventions",
            "Monitor for clinical deterioration"
        ]
    else:
        recommendations = [
            "Standard care protocols appropriate",
            "Routine monitoring sufficient",
            "Continue current treatment plan"
        ]
    
    return {
        'patient_id': patient_idx,
        'risk_score': prediction_proba,
        'risk_category': risk_category,
        'risk_color': risk_color,
        'actual_outcome': 'Mortality' if actual_outcome else 'Survival',
        'patient_values': patient_data.to_dict(),
        'shap_contributions': shap_contributions[:10],
        'lime_explanation': lime_exp.as_list(),
        'recommendations': recommendations
    }

# Generate reports for multiple patients
print("  🔄 Generating reports for sample patients...")

sample_patients = [0, 1, 2, 3, 4]  # First 5 patients
clinical_reports = {}

for idx in sample_patients:
    if idx < len(X_test_scaled):
        # Generate LIME explanation for this patient
        patient_lime = lime_explainer.explain_instance(
            X_test_scaled.iloc[idx].values,
            best_model.predict_proba,
            num_features=8
        )
        
        report = generate_clinical_report(
            idx, best_model, X_test_scaled, y_test, shap_values, patient_lime
        )
        clinical_reports[idx] = report

print(f"✅ Generated clinical reports for {len(clinical_reports)} patients")
</VSCode.Cell>

<VSCode.Cell language="python">
# Display clinical reports in a professional format
print("📊 Displaying Clinical Explanation Reports")
print("="*80)

for patient_id, report in clinical_reports.items():
    print(f"\n{report['risk_color']} PATIENT {patient_id} - CLINICAL EXPLANATION REPORT")
    print("-" * 60)
    
    # Risk Assessment
    print(f"🎯 RISK ASSESSMENT:")
    print(f"   Risk Score: {report['risk_score']:.3f}")
    print(f"   Risk Category: {report['risk_category']}")
    print(f"   Actual Outcome: {report['actual_outcome']}")
    
    # Key Clinical Features
    print(f"\n📊 KEY CONTRIBUTING FACTORS (SHAP Analysis):")
    for i, (feature, contribution) in enumerate(report['shap_contributions'][:5], 1):
        direction = "↗️" if contribution > 0 else "↘️"
        impact = "INCREASES" if contribution > 0 else "DECREASES"
        print(f"   {i}. {feature}: {contribution:+.3f} {direction} {impact} risk")
    
    # LIME Local Explanation
    print(f"\n🔍 LOCAL EXPLANATION (LIME Analysis):")
    for i, (feature_desc, contribution) in enumerate(report['lime_explanation'][:3], 1):
        direction = "↗️" if contribution > 0 else "↘️"
        print(f"   {i}. {feature_desc}: {contribution:+.3f} {direction}")
    
    # Clinical Recommendations
    print(f"\n💡 CLINICAL RECOMMENDATIONS:")
    for i, rec in enumerate(report['recommendations'], 1):
        print(f"   {i}. {rec}")
    
    print("-" * 60)

print(f"\n✅ Clinical explanation reports completed")
</VSCode.Cell>

<VSCode.Cell language="python">
# Create summary dashboard visualization
print("📊 Creating summary dashboard visualization...")

# Prepare data for dashboard
risk_scores = [report['risk_score'] for report in clinical_reports.values()]
risk_categories = [report['risk_category'] for report in clinical_reports.values()]
actual_outcomes = [report['actual_outcome'] for report in clinical_reports.values()]

# Create dashboard with multiple visualizations
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Risk Score Distribution', 'Risk Categories', 'Prediction vs Actual',
        'Top Global Features', 'Model Performance', 'Feature Correlation'
    ],
    specs=[[{"type": "histogram"}, {"type": "pie"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "indicator"}, {"type": "heatmap"}]]
)

# 1. Risk Score Distribution
fig.add_trace(
    go.Histogram(x=risk_scores, nbinsx=10, name="Risk Scores"),
    row=1, col=1
)

# 2. Risk Categories Pie Chart
risk_cat_counts = pd.Series(risk_categories).value_counts()
fig.add_trace(
    go.Pie(labels=risk_cat_counts.index, values=risk_cat_counts.values, name="Risk Categories"),
    row=1, col=2
)

# 3. Prediction vs Actual
pred_actual = pd.DataFrame({
    'Predicted': ['High Risk' if score > 0.5 else 'Low Risk' for score in risk_scores],
    'Actual': actual_outcomes
}).value_counts().reset_index()

fig.add_trace(
    go.Bar(x=pred_actual.iloc[:, 0] + ' | ' + pred_actual.iloc[:, 1], 
           y=pred_actual.iloc[:, 2], name="Pred vs Actual"),
    row=1, col=3
)

# 4. Top Global Features
top_features = importance_df.tail(8)
fig.add_trace(
    go.Bar(x=top_features['importance'], y=top_features['feature'], 
           orientation='h', name="Feature Importance"),
    row=2, col=1
)

# 5. Model Performance Gauge
best_auc = model_results[best_model_name]['auc']
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=best_auc,
        title={"text": "Model AUC"},
        gauge={'axis': {'range': [None, 1]},
               'bar': {'color': "darkblue"},
               'steps': [{'range': [0, 0.7], 'color': "lightgray"},
                        {'range': [0.7, 1], 'color': "gray"}]},
        domain={'x': [0, 1], 'y': [0, 1]}
    ),
    row=2, col=2
)

# 6. Feature Correlation Heatmap (sample)
corr_matrix = X_test_scaled.iloc[:, :6].corr()  # Sample of features
fig.add_trace(
    go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.columns,
               colorscale='RdBu', name="Correlation"),
    row=2, col=3
)

fig.update_layout(
    height=800,
    title_text="Explainable Medical Diagnosis - Clinical Dashboard",
    showlegend=False
)

fig.show()

print("✅ Clinical dashboard visualization created")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Summary and Key Insights

### 🎯 Model Performance
- **Best Model**: {best_model_name}
- **AUC Score**: {model_results[best_model_name]['auc']:.3f}
- **Accuracy**: {model_results[best_model_name]['accuracy']:.3f}
- **F1-Score**: {model_results[best_model_name]['f1_score']:.3f}

### 🔍 Explainability Insights

#### SHAP Analysis (Global Interpretability)
- Provides consistent, game-theory based feature attributions
- Shows which features are most important across all patients
- Reveals feature interactions and non-linear relationships

#### LIME Analysis (Local Interpretability)  
- Explains individual patient predictions with local linear approximations
- Provides intuitive explanations for specific cases
- Complements SHAP with different perspective on feature importance

### 🏥 Clinical Validation
- Model explanations were validated against clinical expectations
- Identified potential areas of concern (bias, data leakage)
- Generated actionable recommendations for clinical use

### 💡 Key Recommendations

1. **For Clinicians**: Use both SHAP and LIME explanations together for comprehensive understanding
2. **For Model Development**: Continuously validate explanations against clinical knowledge
3. **For Deployment**: Implement monitoring for explanation drift and model performance
4. **For Ethics**: Ensure fairness and transparency in AI-assisted clinical decisions

### 🚀 Next Steps
- Implement real-time explanation generation for clinical workflow
- Develop automated alerts for high-risk patients
- Create specialized explanations for different clinical specialties
- Validate with larger datasets and diverse patient populations
</VSCode.Cell>

<VSCode.Cell language="python">
# Final summary and save results
print("💾 Saving analysis results...")

# Create results summary
results_summary = {
    'timestamp': datetime.now().isoformat(),
    'dataset_info': {
        'n_patients': len(X),
        'n_features': len(X.columns),
        'mortality_rate': float(y.mean()),
        'feature_list': list(X.columns)
    },
    'model_performance': {
        'best_model': best_model_name,
        'auc': float(model_results[best_model_name]['auc']),
        'accuracy': float(model_results[best_model_name]['accuracy']),
        'f1_score': float(model_results[best_model_name]['f1_score'])
    },
    'explainability_analysis': {
        'shap_computed': True,
        'lime_computed': True,
        'n_patients_explained': len(clinical_reports),
        'top_features': importance_df.tail(5)['feature'].tolist()
    },
    'clinical_validation': {
        'features_validated': len(clinical_alignment),
        'alignment_rate': aligned_count / total_count if total_count > 0 else 0,
        'concerns_identified': len(concerns)
    }
}

# Save to JSON file
with open('explainable_diagnosis_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

print("✅ Results saved to 'explainable_diagnosis_results.json'")

# Print final summary
print("\n" + "="*80)
print("🎉 EXPLAINABLE MEDICAL DIAGNOSIS ANALYSIS COMPLETED")
print("="*80)
print(f"📊 Dataset: {len(X):,} patients with {len(X.columns)} features")
print(f"🤖 Best Model: {best_model_name} (AUC: {model_results[best_model_name]['auc']:.3f})")
print(f"🔍 Explanations: Generated for {len(clinical_reports)} patients")
print(f"🏥 Clinical Validation: {aligned_count}/{total_count} features aligned with expectations")
print(f"💡 Key Insight: {importance_df.iloc[-1]['feature']} is the most important feature")
print("\n✅ Ready for clinical deployment with explainable AI!")
</VSCode.Cell>
````